# SDP for Generating PPT Entangled States

Implementation of the approach from "A simple class of bound entangled states based on the properties of the antisymmetric subspace" by Sindici & Piani.

**Main Idea**: Generate PPT bound entangled states by solving an SDP that finds the largest probability of obtaining an antisymmetric state from a PPT state.

In [1]:
using LinearAlgebra
using Random
using ProgressMeter
using JLD2
using Ket
using MosekTools
using JuMP
using ppt2

## Helper Functions

In [2]:
# Swap operator: exchanges the two subsystems
function swap(d::Int)
    V = zeros(d, d, d, d)
    for i in 1:d, j in 1:d
        V[j, i, i, j] = 1.0
    end
    return reshape(V, d^2, d^2)
end

# Projector onto symmetric subspace: P_S = (I + V) / 2
function symmetric_projector(d::Int)
    return (I(d^2) + swap(d)) / 2
end

# Projector onto antisymmetric subspace: P_A = (I - V) / 2
function antisymmetric_projector(d::Int)
    return (I(d^2) - swap(d)) / 2
end

# Check if a state is PPT (all eigenvalues of partial transpose are non-negative)
function is_ppt(rho::Matrix, dim_A::Int, dim_B::Int; tol=1e-9)
    return eigmin(Hermitian(partial_transpose(rho, 1))) >= -tol
end

# Check if a state is entangled (has negative eigenvalue under PPT or fails separability test)
function is_entangled(rho::Matrix, dim_A::Int, dim_B::Int; tol=1e-9)
    r, w = entanglement_robustness(rho, [dim_A, dim_B], 2; solver=Mosek.Optimizer)
    return r >= tol
end

is_entangled (generic function with 1 method)

## Random State Generators

In [3]:
# Generate a random quantum state
function rand_state(d::Int; rng=Random.GLOBAL_RNG)
    A = randn(rng, d, d)
    rho = A * A'
    return rho / tr(rho)
end

# Generate a random antisymmetric state
function rand_antisymmetric_state(d::Int; rng=Random.GLOBAL_RNG)
    rho = rand_state(d^2; rng=rng)
    P_A = antisymmetric_projector(d)
    rho_A = P_A * rho * P_A
    rho_A = rho_A / tr(rho_A)
    return rho_A
end

rand_antisymmetric_state (generic function with 1 method)

## SDP Formulation

Solve the SDP from equation (3) in the paper:

$$\max_\sigma \text{Tr}(P_A \sigma)$$

subject to:
- $P_A \sigma P_A = \text{Tr}(P_A \sigma) \rho_A$
- $\sigma \geq 0$
- $\text{Tr}(\sigma) = 1$
- $\sigma^\Gamma \geq 0$ (PPT constraint)

In [4]:
function solve_ppt_sdp(rho_A::Matrix, d::Int; verbose=false)
    """
    Solve the SDP to find the largest probability of obtaining antisymmetric state
    rho_A from a PPT state.

    Returns: (p_opt, sigma_opt) where
    - p_opt: optimal probability
    - sigma_opt: optimal PPT state
    """

    P_A = antisymmetric_projector(d)
    P_S = symmetric_projector(d)

    model = Model(Mosek.Optimizer)
    if !verbose
        set_silent(model)
    end

    # Decision variable: sigma is a positive semidefinite matrix
    @variable(model, sigma[1:d^2, 1:d^2], PSD)
    @variable(model, p >= 0)  # probability variable

    # Objective: maximize p
    @objective(model, Max, p)

    # Constraint 1: Tr(sigma) = 1
    @constraint(model, tr(sigma) == 1)

    # Constraint 2: P_A sigma P_A = p * rho_A
    # This is implemented as: P_A sigma P_A - p * rho_A = 0
    @constraint(model, P_A * sigma * P_A .== p .* rho_A)

    # Constraint 3: sigma is PPT
    sigma_PT = partial_transpose(sigma, 1)
    @constraint(model, sigma_PT in PSDCone())

    optimize!(model)

    status = termination_status(model)
    if status != OPTIMAL
        @warn "SDP did not converge optimally: $status"
    end

    p_opt = value(p)
    sigma_opt = value.(sigma)

    return p_opt, sigma_opt
end

solve_ppt_sdp (generic function with 1 method)

## Lower Bound on p^PPT

From Theorem 1 in the paper, for any antisymmetric state $\rho_A$:
$$\frac{2}{d(d+1)+2} \leq p^{PPT}(\rho_A) \leq \frac{1}{2}$$

The lower bound is achieved by the family of states:
$$\sigma(p) = p \rho_A \oplus (1-p) \frac{P_S}{d_S}$$

In [5]:
# Lower bound from the theorem: 2 / (d(d+1) + 2)
function lower_bound_ppt(d::Int)
    d_S = d * (d + 1) ÷ 2
    return 2 / (d * (d + 1) + 2)
end

# Upper bound: 1/2 for all antisymmetric states
upper_bound_ppt(d::Int) = 0.5

# Family of PPT states: sigma(p) = p * rho_A ⊕ (1-p) * P_S / d_S
function construct_ppt_family(rho_A::Matrix, d::Int, p::Real)
    P_S = symmetric_projector(d)
    d_S = d * (d + 1) ÷ 2
    
    sigma = p .* rho_A .+ (1 - p) .* P_S ./ d_S
    sigma = sigma / tr(sigma)
    return sigma
end

construct_ppt_family (generic function with 1 method)

## Main Procedure for Generating PPT Entangled States

In [6]:
function generate_ppt_entangled_state(d::Int; rng=Random.GLOBAL_RNG, verbose=false, max_attempts=100)
    """
    Generate a PPT entangled state using the procedure from the paper:
    1. Generate random antisymmetric state rho_A
    2. Solve SDP to find p^PPT(rho_A)
    3. If p^PPT(rho_A) < 1/2, the optimal PPT state is entangled

    Returns: (sigma, rho_A, p_opt) or (nothing, nothing, nothing) if unsuccessful
    """

    for attempt in 1:max_attempts
        if verbose
            println("Attempt $attempt/$max_attempts")
        end

        # Step 1: Generate random antisymmetric state
        rho_A = rand_antisymmetric_state(d; rng=rng)

        # Step 2: Solve SDP
        p_opt, sigma_opt = solve_ppt_sdp(rho_A, d; verbose=verbose)

        # Check if state is PPT
        if !is_ppt(sigma_opt, d, d)
            if verbose
                println("  State is not PPT (numerical error?)")
            end
            continue
        end

        # Step 3: Check if p < 1/2 (which implies entanglement by Lemma 1)
        if p_opt < 0.5 - 1e-6
            if verbose
                println("  Found PPT entangled state! p_opt = $p_opt")
            end

            # Verify entanglement
            if is_entangled(sigma_opt, d, d)
                return sigma_opt, rho_A, p_opt
            end
        end
    end

    return nothing, nothing, nothing
end

generate_ppt_entangled_state (generic function with 1 method)

## Example: Generate PPT Entangled State for d=3

In [9]:
d = 4
rng = Xoshiro(42)

println("Generating PPT entangled states for d = $d")
println("Lower bound on p^PPT: $(lower_bound_ppt(d))")
println("Upper bound on p^PPT: $(upper_bound_ppt(d))")
println()

Generating PPT entangled states for d = 4
Lower bound on p^PPT: 0.09090909090909091
Upper bound on p^PPT: 0.5



In [10]:
sigma, rho_A, p_opt = generate_ppt_entangled_state(d; rng=rng, verbose=false, max_attempts=5000)

if sigma !== nothing
    println("\n=== SUCCESS ===")
    println("p_opt = $p_opt")
    println("Is PPT: $(is_ppt(sigma, d, d))")
    println("Is entangled: $(is_entangled(sigma, d, d))")
    
    # Verify constraint: P_A sigma P_A = p_opt * rho_A
    P_A = antisymmetric_projector(d)
    lhs = P_A * sigma * P_A
    rhs = p_opt .* rho_A
    constraint_error = norm(lhs - rhs)
    println("Constraint P_A σ P_A = p*ρ_A error: $constraint_error")
else
    println("\nFailed to generate PPT entangled state in max attempts.")
end


=== SUCCESS ===
p_opt = 0.49829905579777806
Is PPT: true
Is entangled: true
Constraint P_A σ P_A = p*ρ_A error: 7.436520072943022e-12


In [21]:
eigmin(sigma), eigmin(partial_transpose(sigma, 1))

(8.44029847512464e-11, 4.172286816761532e-13)

In [35]:
r, w = entanglement_robustness(Hermitian(ampliation(sigma, sigma, d, d)), [d, d], 2; solver=Mosek.Optimizer)

(-0.005468126233903355, [1.5822705840241077e-7 7.242956407902949e-6 … -0.01683082146620338 -0.11172248709331128; 7.242956407902949e-6 0.19722683076609135 … 0.008403536602399874 0.055782510245989625; … ; -0.01683082146620338 0.008403536602399874 … 0.00042203990033605796 2.4250597897191386e-7; -0.11172248709331128 0.055782510245989625 … 2.4250597897191386e-7 1.348699185423664e-7])

In [34]:
w

16×16 Symmetric{Float64, Matrix{Float64}}:
  1.58227e-7  7.24296e-6   8.71023e-7  …  -0.0168308   -0.111722
  7.24296e-6  0.197227     0.0223624       0.00840354   0.0557825
  8.71023e-7  0.0223624    0.00253559      0.00103429   0.00686582
  5.58773e-6  0.148441     0.0168308       2.96071e-7   2.23718e-6
 -7.49088e-6  2.27092e-8  -6.63615e-6      0.133796     0.0557854
 -0.197242    4.16404e-7  -0.177768    …  -0.0668034   -0.0278534
 -0.022364    3.36845e-8  -0.0201561      -0.00822223  -0.00342823
 -0.148452    3.68584e-7  -0.133796       -2.56728e-6  -1.07144e-6
 -8.03821e-7  6.65165e-6   1.0075e-8       3.75101e-6   0.00685828
 -0.022367    0.177762    -4.94329e-6     -1.88288e-6  -0.0034243
 -0.00253606  0.0201553   -5.48056e-7  …  -2.62149e-7  -0.000421387
 -0.0168344   0.133791    -3.78637e-6     -3.30886e-8  -2.72877e-8
 -5.51224e-6  2.77085e-6   4.10656e-7     -0.00686572  -2.11985e-6
 -0.148441    0.0741157    0.00912212      0.00342806   1.10363e-6
 -0.0168308   0.00840354

In [22]:
eigmin(Hermitian(rho_A)), eigmin(partial_transpose(rho_A, 1))

(0.0, -0.2626422528932632)

## Batch Generation and Analysis

In [44]:
function batch_generate_ppt_entangled_states(d::Int, n_samples::Int; rng=Random.GLOBAL_RNG)
    """
    Generate multiple PPT entangled states and collect statistics.
    """
    states = []
    p_values = []
    success_count = 0

    @showprogress for i in 1:n_samples
        sigma, rho_A, p_opt = generate_ppt_entangled_state(d; rng=rng, max_attempts=100)
        if sigma !== nothing
            push!(states, sigma)
            push!(p_values, p_opt)
            success_count += 1
        end
    end

    return states, p_values, success_count
end

batch_generate_ppt_entangled_states (generic function with 1 method)

In [47]:
d = 4
n = 100
states, p_values, success = batch_generate_ppt_entangled_states(d, n; rng=rng)

println("\n=== Batch Generation Results ===")
println("Dimension: $d")
println("Successful states: $success / $n")
if length(p_values) > 0
    println("p^PPT values:")
    println("  Min: $(minimum(p_values))")
    println("  Max: $(maximum(p_values))")
    println("  Mean: $(mean(p_values))")
    println("  Std: $(std(p_values))")
end

Progress: 100%|█████████████████████████████████████████| Time: 0:02:04



=== Batch Generation Results ===
Dimension: 4
Successful states: 2 / 100
p^PPT values:
  Min: 0.49907330003157163
  Max: 0.4996093015189203


UndefVarError: UndefVarError: `mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name also exists in Statistics.
    - Also exported by BenchmarkTools (loaded but not imported in Main).

## Verify Theoretical Bounds

In [214]:
# Test lower bound construction for different dimensions
for d in [2, 3, 4]
    P_S = symmetric_projector(d)
    d_S = d * (d + 1) ÷ 2
    p_bar = lower_bound_ppt(d)
    
    rho_A = rand_antisymmetric_state(d; rng=rng)
    sigma = construct_ppt_family(rho_A, d, p_bar * 0.99)
    
    is_ppt_state = is_ppt(sigma, d, d)
    
    println("d = $d:")
    println("  Lower bound p_bar = $p_bar")
    println("  σ(0.99*p_bar) is PPT: $is_ppt_state")
    println()
end

d = 2:
  Lower bound p_bar = 0.25
  σ(0.99*p_bar) is PPT: true

d = 3:
  Lower bound p_bar = 0.14285714285714285
  σ(0.99*p_bar) is PPT: true

d = 4:
  Lower bound p_bar = 0.09090909090909091
  σ(0.99*p_bar) is PPT: true

